In [117]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import math
from collections import Counter

from ultralytics.utils.checks import check_imshow
from ultralytics.utils.plotting import Annotator, colors

from collections import defaultdict

In [118]:
# COLORS = sv.ColorPalette.from_hex(["#E6194B", "#3CB44B", "#FFE119", "#3C76D1"])
COLORS = sv.ColorPalette.from_hex(["#00FF00", "#FF0000"])

ZONE_IN_POLYGONS = [
    np.array([[783, 1041], [575, 1017], [831, 836], [1002, 925]]),
    np.array([[1510, 579], [1509, 509], [1902, 646], [1886, 765]]),
    np.array([[1169, 356], [1068, 310], [1225, 204], [1325, 242]]),
    np.array([[582, 451], [492, 484], [353, 388], [481, 361]]),
]

ZONE_OUT_POLYGONS = [
    np.array([[339, 940], [472, 993], [604, 763], [433, 730]]),
    np.array([[1491, 830], [1500, 732], [1869, 772], [1844, 885]]), ########
    np.array([[1263, 355], [1350, 375], [1408, 251], [1342, 230]]),
    np.array([[673, 388], [705, 335], [459, 270], [419, 342]]),
]

model = YOLO("y11m-lh-sinvan.pt")
class_names = model.model.names

names = list(class_names.values())
for i in range(len(names)):
    names.append("indeterminado")

# inicializo en cero los dos arrays (de entrada y salida) que tendrán la cantidad de objetos finales por zona y por clase
"""
Example:
    {
        0: {
            "bicycle": 0,
            "bus": 0,
            "car": 0,
            "motorbike": 0,
            "truck": 0,
            "van": 0
        },
        .
        .
        .
        3: {
            "bicycle": 0,
            "bus": 0,
            "car": 0,
            "motorbike": 0,
            "truck": 0,
            "van": 0
        }
    }
"""
total_obj_zone_in = { i: {key: 0 for key in names} for i in range(len(ZONE_IN_POLYGONS)) }
total_obj_zone_out = { i: {key: 0 for key in names} for i in range(len(ZONE_OUT_POLYGONS)) }

"""
classes = {
    "car": "Auto",
    "bus": "Colectivo",
    "light_truck": "Camión liviano",
    "heavy_truck": "Camión pesado",
    "motorbike": "Moto",
    "bicycle": "Bicicleta",
}
"""

'\nclasses = {\n    "car": "Auto",\n    "bus": "Colectivo",\n    "light_truck": "Camión liviano",\n    "heavy_truck": "Camión pesado",\n    "motorbike": "Moto",\n    "bicycle": "Bicicleta",\n}\n'

In [119]:
obj_zones_in = []
obj_zones_out = []
obj_in_for_zones = defaultdict(lambda: [])
obj_in_out_zones = defaultdict(lambda: [])
track_results = defaultdict(lambda: [])

track_history = defaultdict(lambda: [])
data_obj_history = defaultdict(lambda: [])

def get_center_bb(box):
    x_center = int((box[0] + box[2]) / 2)
    y_center = int((box[1] + box[3]) / 2)
    return (x_center, y_center)

def draw_polygons(annotated_frame, polygon, number_polygon, zone_type, thickness):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    cv2.polylines(
        annotated_frame, [polygon], isClosed=True, color=COLORS.colors[zone_type].as_bgr(), thickness=thickness
    )
    zone_center = sv.get_polygon_center(polygon=polygon)
    cv2.putText(annotated_frame, str(number_polygon), (int(zone_center.x), int(zone_center.y)), font, font_scale, COLORS.colors[zone_type].as_bgr(), thickness=thickness)
    
    return annotated_frame

def draw_zones_in_out(annotated_frame, thickness):
    for i, (zone_in, zone_out) in enumerate(zip(ZONE_IN_POLYGONS, ZONE_OUT_POLYGONS)):
        draw_polygons(annotated_frame, zone_in, i, 0, thickness)
    return annotated_frame

def detect_zone_in(box):
    for i, (polygon) in enumerate(ZONE_IN_POLYGONS):
        if (cv2.pointPolygonTest(polygon, get_center_bb(box), False) > 0):  # > 0 dentro del polígono
            return i

    return -1

def save_zone_in(box, track_id):
    if track_id not in obj_zones_in:
        zone_in = detect_zone_in(box)
        if zone_in >= 0:
            obj_zones_in.append(track_id)
            obj_in_for_zones[zone_in].append(track_id)

                
def draw_bb_and_save_track(frame, annotator, box, cls, track_id, act_frame, confidence):
    annotator.box_label(box, color=colors(int(cls), True), label=f"{track_id} - {class_names[int(cls)]}")

    # Store tracking and data object history
    data_obj_history[track_id].append(
        {
            "act_frame": act_frame,
            "class_id": int(cls),
            "confidence": confidence
        })
    track = track_history[track_id]
    track.append((int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2)))
    if len(track) > 30:
        track.pop(0)

    # Plot tracks
    points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(frame, [points], isClosed=False, color=colors(int(cls), True), thickness=2)


H_UMBRAL = 0.8  # Ajusta según tu necesidad

def calculate_entropy(track_data, track_id):
    total_track = len(track_data)
    class_counts = Counter()
    for item in track_data:
        class_counts[item['class_id']] += 1
    probabilities = [count / total_track for class_id, count in class_counts.items()]
    entropy = -sum(p * math.log2(p) for p in probabilities if p > 0)
    
    return class_counts, entropy

def classify_track(entropy, class_counts):
    """Determina la clase del trackeo o si hay incertidumbre."""
    assigned_class = max(class_counts, key=class_counts.get)
    if entropy < H_UMBRAL:
        # Si la entropía es baja, asignar la clase con mayor confianza promedio
        return class_names[assigned_class]
    else:
        # return "probably_" + class_names[assigned_class]
        return "indeterminado"

def get_final_results():
    for track_id, data in data_obj_history.items():
        class_counts, entropy = calculate_entropy(data, track_id)
        classification = classify_track(entropy, class_counts)
        track_results[track_id] = {
            "entropy": entropy,
            "classification": classification
        }
    if (track_id == 51):
        print("ACA")
    for track_id, result in track_results.items():
        print(f"Track {track_id} -> Entropía: {result['entropy']:.4f}, Clasificación: {result['classification']}")


In [120]:
video_path = "vuelo01_1920.mp4"
cap = cv2.VideoCapture(video_path)

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# result = cv2.VideoWriter("object_tracking.avi",
#                        cv2.VideoWriter_fourcc(*'mp4v'),
#                        fps,
#                        (w, h))

# inicializo en cero los arays que tendrán la cantidad de objetos finales por zona
act_frame = 0

while cap.isOpened():
    success, frame = cap.read()

    act_frame += 1
    if act_frame % 2 == 0:
        if success:
            results = model.track(frame, persist=True, verbose=False)
            boxes = results[0].boxes.xyxy.cpu()

            draw_zones_in_out(frame, 2)
            
            if results[0].boxes.id is not None:
                clss = results[0].boxes.cls.cpu().tolist()
                track_ids = results[0].boxes.id.int().cpu().tolist()
                confs = results[0].boxes.conf.float().cpu().tolist()
                # Annotator Init
                annotator = Annotator(frame, line_width=1)
                for box, cls, track_id, confidence in zip(boxes, clss, track_ids, confs):
                    save_zone_in(box, track_id)
                    # if (track_id in obj_in_out_zones):
                    draw_bb_and_save_track(frame, annotator, box, cls, track_id, act_frame, confidence)

            cv2.imshow("Video", frame)
            # result.write(frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        else:
            break

# print(total_obj_zone_in)
# print(total_obj_zone_out)
# result.release()
cap.release()
cv2.destroyAllWindows()

get_final_results()

Track 1 -> Entropía: -0.0000, Clasificación: car
Track 2 -> Entropía: 0.3966, Clasificación: car
Track 3 -> Entropía: -0.0000, Clasificación: car
Track 4 -> Entropía: 0.2387, Clasificación: car
Track 5 -> Entropía: -0.0000, Clasificación: car
Track 6 -> Entropía: -0.0000, Clasificación: car
Track 7 -> Entropía: -0.0000, Clasificación: car
Track 8 -> Entropía: -0.0000, Clasificación: car
Track 9 -> Entropía: -0.0000, Clasificación: car
Track 10 -> Entropía: -0.0000, Clasificación: car
Track 11 -> Entropía: -0.0000, Clasificación: car
Track 12 -> Entropía: -0.0000, Clasificación: car
Track 13 -> Entropía: -0.0000, Clasificación: car
Track 14 -> Entropía: -0.0000, Clasificación: car
Track 15 -> Entropía: -0.0000, Clasificación: car
Track 16 -> Entropía: -0.0000, Clasificación: car
Track 17 -> Entropía: -0.0000, Clasificación: bus
Track 18 -> Entropía: 0.7140, Clasificación: car
Track 19 -> Entropía: 0.2580, Clasificación: car
Track 20 -> Entropía: -0.0000, Clasificación: motorbike
Track 2

In [121]:
print(obj_in_for_zones.items())
for zone, tracks in obj_in_for_zones.items():
    for track in tracks:
        print(track)
        total_obj_zone_in[zone][track_results[track]["classification"]] += 1
    # print(f"    {zone}: {tracks}")
print(total_obj_zone_in)

dict_items([(0, [1, 19, 41]), (1, [5, 9, 15, 47]), (3, [14]), (2, [37])])
1
19
41
5
9
15
47
14
37
{0: {'bicycle': 0, 'bus': 0, 'car': 3, 'heavy_truck': 0, 'light_truck': 0, 'motorbike': 0, 'indeterminado': 0}, 1: {'bicycle': 0, 'bus': 0, 'car': 4, 'heavy_truck': 0, 'light_truck': 0, 'motorbike': 0, 'indeterminado': 0}, 2: {'bicycle': 0, 'bus': 0, 'car': 0, 'heavy_truck': 0, 'light_truck': 0, 'motorbike': 0, 'indeterminado': 1}, 3: {'bicycle': 0, 'bus': 0, 'car': 1, 'heavy_truck': 0, 'light_truck': 0, 'motorbike': 0, 'indeterminado': 0}}


In [122]:
for i, (zone) in enumerate(total_obj_zone_in):
    print(f"Zona de entrada {i}")
    for key, value in total_obj_zone_in[i].items():
        print(f"    {key}: {value}")

for i, (zone) in enumerate(total_obj_zone_out):
    print(f"Zona de salida {i}")
    for key, value in total_obj_zone_out[i].items():
        print(f"    {key}: {value}")

Zona de entrada 0
    bicycle: 0
    bus: 0
    car: 3
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 0
Zona de entrada 1
    bicycle: 0
    bus: 0
    car: 4
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 0
Zona de entrada 2
    bicycle: 0
    bus: 0
    car: 0
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 1
Zona de entrada 3
    bicycle: 0
    bus: 0
    car: 1
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 0
Zona de salida 0
    bicycle: 0
    bus: 0
    car: 0
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 0
Zona de salida 1
    bicycle: 0
    bus: 0
    car: 0
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 0
Zona de salida 2
    bicycle: 0
    bus: 0
    car: 0
    heavy_truck: 0
    light_truck: 0
    motorbike: 0
    indeterminado: 0
Zona de salida 3
    bicycle: 0
    bus: 0
    car: 0
    heavy_truck: 0
    light_tru